# Tutorial: Custom Memory + Azure AI Search

This notebook builds a custom memory layer using only the Agent Framework `ContextProvider` and Azure AI Search.

The goals are:

| Area | Design in this notebook |
|---|---|
| Memory store | Custom Azure AI Search index |
| Retrieval modes | Choose keyword, vector, hybrid, or semantic hybrid retrieval |
| Ranking profiles | Choose no boost, importance-oriented ranking, or recency-oriented ranking |
| Sorting | Use top-level fields such as `created_at`, `updated_at`, `importance`, and `last_accessed_at` |
| Memory extraction | Ask an LLM to extract memory candidates as JSON. Use a rule-based fallback when the LLM is unavailable |
| Memory consolidation | Search for nearby existing memories in Azure AI Search, then decide `ADD` / `UPDATE` / `DELETE` / `NONE` with an LLM or rule fallback |
| Human-like memory control | Track importance, access count, last access, expiration, decay, and archival state |

Memory extraction and consolidation are implemented as explicit classes. This keeps the search schema and memory lifecycle fully controllable in Azure AI Search.

## 1. Dependencies

This notebook uses `azure-search-documents`, `azure-identity`, `openai`, and `python-dotenv`. The Agent Framework `ContextProvider` is loaded from the local package included in this workshop when available.

In [ ]:
# Uncomment and run if needed.
# %pip install -q "azure-search-documents==11.5.2" azure-identity openai python-dotenv

print("Dependency cell checked. If packages are missing, run the %pip install line above.")

## 2. Environment Variables and Settings

Azure AI Search is used as the memory store. Azure OpenAI is used for memory extraction, consolidation decisions, and embeddings.

In [ ]:
# pyright: reportMissingImports=false, reportUnusedImport=false, reportUnusedVariable=false

import json
import logging
import math
import os
import re
import sys
import uuid
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from enum import StrEnum
from pathlib import Path
from typing import Any

from dotenv import load_dotenv

load_dotenv()

LOCAL_AGENT_FRAMEWORK_PACKAGES = Path("agent-framework-python-1.4.0/python/packages")
for package_name in ["core", "openai", "azure-ai-search"]:
    package_path = LOCAL_AGENT_FRAMEWORK_PACKAGES / package_name
    if package_path.exists():
        sys.path.insert(0, str(package_path.resolve()))

# === Azure AI Search ===
AZURE_SEARCH_ENDPOINT = os.getenv("AZURE_SEARCH_ENDPOINT")
AZURE_SEARCH_API_KEY = os.getenv("AZURE_SEARCH_API_KEY")
AZURE_SEARCH_INDEX_NAME = os.getenv("CUSTOM_MEMORY_SEARCH_INDEX_NAME", "custom-memory")

# === Azure OpenAI ===
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION", "2024-10-21")
AZURE_OPENAI_CHAT_DEPLOYMENT = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")
AZURE_OPENAI_EMBEDDING_DEPLOYMENT = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT_NAME")
AZURE_OPENAI_EMBEDDING_DIMS = int(os.getenv("AZURE_OPENAI_EMBEDDING_DIMS", "1536"))

# === Demo IDs ===
DEMO_USER_ID = os.getenv("CUSTOM_MEMORY_DEMO_USER_ID", "workshop-user")
DEMO_AGENT_ID = os.getenv("CUSTOM_MEMORY_DEMO_AGENT_ID", "travel-agent")

# === Authentication mode ===
# True: API key auth, False: DefaultAzureCredential auth
USE_KEY_AUTH = os.getenv("USE_KEY_AUTH", "False").lower() in ("true", "1", "t")

if USE_KEY_AUTH:
    print("Auth mode: API key")
else:
    os.environ.pop("AZURE_OPENAI_API_KEY", None)
    print("Auth mode: DefaultAzureCredential")

# === Readiness ===
SEARCH_READY = bool(AZURE_SEARCH_ENDPOINT and AZURE_SEARCH_INDEX_NAME)
LLM_READY = bool(AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_CHAT_DEPLOYMENT)
EMBEDDING_READY = bool(AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_EMBEDDING_DEPLOYMENT)

print(f"\nSEARCH_READY: {SEARCH_READY}")
print(f"LLM_READY: {LLM_READY}")
print(f"EMBEDDING_READY: {EMBEDDING_READY}")

# === Logger ===
custom_memory_logger = logging.getLogger("custom_memory.aisearch")
custom_memory_logger.setLevel(logging.INFO)
if not custom_memory_logger.handlers:
    _handler = logging.StreamHandler()
    _handler.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(name)s - %(message)s"))
    custom_memory_logger.addHandler(_handler)
custom_memory_logger.propagate = False

### 2.1. Set Up OpenTelemetry Tracing

OpenTelemetry tracing is useful when debugging multi-agent behavior.

In this environment, `setup_observability` is not available in the Agent Framework package, so this notebook manually configures OpenTelemetry providers and enables instrumentation.

The trace destination is OTLP over gRPC, such as Jaeger or an OpenTelemetry Collector on port `4317`.

Jaeger UI: [http://localhost:16686](http://localhost:16686)

In [ ]:
import os
from contextlib import nullcontext


class NoOpSpan:
    def set_attribute(self, _name, _value) -> None:
        del _name, _value
        return None


class NoOpTracer:
    def start_as_current_span(self, _span_name):
        del _span_name
        return nullcontext(NoOpSpan())


service_name = os.getenv("OTEL_SERVICE_NAME", "custom-memory-aisearch-notebook")
otlp_endpoint = os.getenv("OTEL_EXPORTER_OTLP_ENDPOINT", "http://localhost:4317")

os.environ.setdefault("OTEL_SERVICE_NAME", service_name)
os.environ.setdefault("OTEL_EXPORTER_OTLP_TRACES_ENDPOINT", otlp_endpoint)
os.environ.setdefault("OTEL_EXPORTER_OTLP_LOGS_ENDPOINT", otlp_endpoint)
os.environ.setdefault("OTEL_EXPORTER_OTLP_PROTOCOL", "grpc")

try:
    from agent_framework.observability import configure_otel_providers, get_tracer

    if not globals().get("CUSTOM_MEMORY_OTEL_CONFIGURED", False):
        configure_otel_providers(
            enable_sensitive_data=os.getenv("CUSTOM_MEMORY_TRACE_SENSITIVE_DATA", "true").casefold() in {"1", "true", "yes", "y"},
        )
        CUSTOM_MEMORY_OTEL_CONFIGURED = True
    tracer = get_tracer()
    print(f"Observability configured: service={service_name} otlp={otlp_endpoint}")
except Exception as error:
    tracer = NoOpTracer()
    CUSTOM_MEMORY_OTEL_CONFIGURED = False
    print("Could not initialize OpenTelemetry. Using a no-op tracer:", error)

## 3. Memory Schema and Lifecycle

This section defines the core data model for treating extracted conversation facts as searchable and forgettable memories. Saving only the memory text would be simple, but memories injected into an agent directly affect answer quality. For that reason, we store fields that support retrieval, filtering, ranking, and forgetting from the beginning.

| Definition | Role | Why it matters |
|---|---|---|
| `MemoryRecord` | One persisted memory document in Azure AI Search | Stores the text together with importance, status, timestamps, and access count |
| `MemoryStatus` | `active`, `archived`, or `deleted` | Keeps old or deleted memories out of the default retrieval path |
| `SearchMode` | Switches keyword, vector, hybrid, and semantic retrieval | Makes the candidate retrieval algorithm explicit |
| `RankingProfile` | Switches importance boost, recency boost, or no boost | Selects an Azure AI Search scoring profile independently from the retrieval mode |
| `MemorySearchOptions` | Groups search-time options | Lets callers specify retrieval mode, ranking profile, result count, category, and visibility of archived memories |
| `HumanMemoryPolicy` | Recall, decay, and forgetting rules | Makes recalled memories easier to keep and weak old memories gradually less visible |

If everything is hidden inside a free-form JSON payload, such as Mem0-style `payload`, Azure AI Search cannot easily filter, sort, or boost those values. This notebook promotes `created_at`, `updated_at`, `importance`, `retention_score`, and `access_count` to top-level fields so Azure AI Search can use them directly for ranking and filtering.

In [ ]:
class MemoryStatus(StrEnum):
    ACTIVE = "active"
    ARCHIVED = "archived"
    DELETED = "deleted"


class SearchMode(StrEnum):
    KEYWORD = "keyword"
    VECTOR = "vector"
    HYBRID = "hybrid"
    SEMANTIC = "semantic"


class RankingProfile(StrEnum):
    NONE = "none"
    IMPORTANT = "important"
    RECENT = "recent"


class MemoryEvent(StrEnum):
    ADD = "ADD"
    UPDATE = "UPDATE"
    DELETE = "DELETE"
    NONE = "NONE"


MEMORY_PRIORITY_SCORING_PROFILE = "memory-priority"
MEMORY_RECENCY_SCORING_PROFILE = "memory-recency"


def utc_now() -> datetime:
    return datetime.now(timezone.utc)


def to_utc_datetime(value: datetime | str | None) -> datetime | None:
    if value is None:
        return None
    if isinstance(value, datetime):
        parsed = value
    else:
        try:
            parsed = datetime.fromisoformat(value.replace("Z", "+00:00"))
        except ValueError:
            return None
    if parsed.tzinfo is None:
        parsed = parsed.replace(tzinfo=timezone.utc)
    return parsed.astimezone(timezone.utc)


def dt_to_json(value: datetime | None) -> str | None:
    return value.isoformat() if value else None


def normalize_text(value: str) -> str:
    return re.sub(r"\s+", " ", value.strip()).casefold()


@dataclass
class MemoryCandidate:
    memory: str
    category: str = "general"
    importance: float = 0.5
    confidence: float = 0.7
    source: str = "conversation"
    expires_at: datetime | None = None
    metadata: dict[str, Any] = field(default_factory=dict)


@dataclass
class MemoryRecord:
    id: str
    user_id: str
    agent_id: str
    memory: str
    category: str = "general"
    summary: str = ""
    importance: float = 0.5
    confidence: float = 0.7
    status: MemoryStatus = MemoryStatus.ACTIVE
    source: str = "conversation"
    created_at: datetime = field(default_factory=utc_now)
    updated_at: datetime = field(default_factory=utc_now)
    last_accessed_at: datetime | None = None
    expires_at: datetime | None = None
    access_count: int = 0
    retention_score: float = 1.0
    metadata: dict[str, Any] = field(default_factory=dict)

    def to_search_document(self, embedding: list[float] | None = None) -> dict[str, Any]:
        document = {
            "id": self.id,
            "user_id": self.user_id,
            "agent_id": self.agent_id,
            "memory": self.memory,
            "summary": self.summary or self.memory,
            "category": self.category,
            "importance": float(self.importance),
            "confidence": float(self.confidence),
            "status": str(self.status),
            "source": self.source,
            "created_at": self.created_at,
            "updated_at": self.updated_at,
            "last_accessed_at": self.last_accessed_at,
            "expires_at": self.expires_at,
            "access_count": int(self.access_count),
            "retention_score": float(self.retention_score),
            "metadata_json": json.dumps(self.metadata, ensure_ascii=False),
        }
        if embedding is not None:
            document["embedding"] = embedding
        return document

    @classmethod
    def from_search_document(cls, document: dict[str, Any]) -> "MemoryRecord":
        metadata_raw = document.get("metadata_json") or "{}"
        try:
            metadata = json.loads(metadata_raw)
        except json.JSONDecodeError:
            metadata = {"raw_metadata": metadata_raw}
        status_value = str(document.get("status") or MemoryStatus.ACTIVE)
        return cls(
            id=str(document["id"]),
            user_id=str(document.get("user_id") or ""),
            agent_id=str(document.get("agent_id") or ""),
            memory=str(document.get("memory") or ""),
            category=str(document.get("category") or "general"),
            summary=str(document.get("summary") or ""),
            importance=float(document.get("importance") or 0.5),
            confidence=float(document.get("confidence") or 0.7),
            status=MemoryStatus(status_value),
            source=str(document.get("source") or "conversation"),
            created_at=to_utc_datetime(document.get("created_at")) or utc_now(),
            updated_at=to_utc_datetime(document.get("updated_at")) or utc_now(),
            last_accessed_at=to_utc_datetime(document.get("last_accessed_at")),
            expires_at=to_utc_datetime(document.get("expires_at")),
            access_count=int(document.get("access_count") or 0),
            retention_score=float(document.get("retention_score") or 1.0),
            metadata=metadata,
        )


@dataclass
class MemorySearchOptions:
    mode: SearchMode = SearchMode.HYBRID
    ranking_profile: RankingProfile = RankingProfile.IMPORTANT
    top: int = 5
    order_by: str | None = None
    category: str | None = None
    include_archived: bool = False
    min_importance: float | None = None
    min_retention_score: float | None = None
    semantic_configuration_name: str = "memory-semantic"


@dataclass
class MemorySearchResult:
    record: MemoryRecord
    score: float | None = None
    reranker_score: float | None = None


@dataclass
class ConsolidationDecision:
    event: MemoryEvent
    candidate: MemoryCandidate
    target_id: str | None = None
    merged_memory: str | None = None
    reason: str = ""


class HumanMemoryPolicy:
    def __init__(
        self,
        *,
        half_life_days: float = 45.0,
        access_boost: float = 0.04,
        archive_threshold: float = 0.22,
    ):
        self.half_life_days = half_life_days
        self.access_boost = access_boost
        self.archive_threshold = archive_threshold

    def retention_score(self, record: MemoryRecord, now: datetime | None = None) -> float:
        now = now or utc_now()
        age_days = max((now - record.updated_at).total_seconds() / 86400, 0.0)
        decay = 0.5 ** (age_days / self.half_life_days)
        access_rehearsal = min(record.access_count * self.access_boost, 0.35)
        importance_anchor = 0.35 + (0.65 * record.importance)
        if record.expires_at and record.expires_at <= now:
            return 0.0
        return max(0.0, min(1.0, decay * importance_anchor + access_rehearsal))

    def status_for(self, record: MemoryRecord, now: datetime | None = None) -> MemoryStatus:
        score = self.retention_score(record, now=now)
        if score <= self.archive_threshold:
            return MemoryStatus.ARCHIVED
        return MemoryStatus.ACTIVE

    def touch(self, record: MemoryRecord, now: datetime | None = None) -> MemoryRecord:
        now = now or utc_now()
        record.last_accessed_at = now
        record.access_count += 1
        record.retention_score = self.retention_score(record, now=now)
        if record.status != MemoryStatus.DELETED:
            record.status = self.status_for(record, now=now)
        return record

print("Defined memory schema and HumanMemoryPolicy.")

### How to Use `MemorySearchOptions`

`MemorySearchOptions` describes which retrieval mode to use, which ranking profile to apply, and how many memories should be visible to the agent. Memory search needs more than nearest-neighbor text matching: stale memories, deleted memories, and unrelated categories can all lead the agent to answer from the wrong premise.

The usual pattern is to pass `options=MemorySearchOptions(...)` when searching. `mode` controls how candidates are retrieved. `ranking_profile` controls how Azure AI Search scoring profiles adjust the ranking. For example, combining `SearchMode.HYBRID` and `RankingProfile.IMPORTANT` retrieves candidates with keyword + vector search, then boosts importance and retention signals.

```python
options = MemorySearchOptions(
    mode=SearchMode.HYBRID,
    ranking_profile=RankingProfile.IMPORTANT,
    top=5,
    min_retention_score=0.1,
)
results = custom_memory_provider.search_memories("Osaka trip preferences", options=options)
```

Key fields:

| Field | Purpose | Example |
|---|---|---|
| `mode` | Selects the retrieval algorithm | Use `SearchMode.KEYWORD` for exact terms and `SearchMode.HYBRID` when semantic similarity also matters |
| `ranking_profile` | Selects the ranking policy for the candidate set | Use `RankingProfile.IMPORTANT` for durable constraints, `RankingProfile.RECENT` for fresh conditions, and `RankingProfile.NONE` for comparison |
| `top` | Controls how many results are returned | Keep agent context small, often 3 to 5 memories |
| `category` | Filters by memory type | Search travel, food, or work settings separately |
| `include_archived` | Includes archived memories | Use only for audits or forgetting checks |
| `min_importance` | Filters out low-importance memories | Use when only strong memories should affect the answer |
| `min_retention_score` | Filters out low-retention memories | Keeps stale and unreliable memories out of default search |
| `order_by` | Uses explicit sorting instead of relevance ranking | Use `updated_at desc` for audit views |

Callers do not need to pass raw Azure AI Search scoring profile names. `RankingProfile.IMPORTANT` maps to `memory-priority`; `RankingProfile.RECENT` maps to `memory-recency`. Azure AI Search scoring profiles can be used with keyword, vector, and hybrid queries, but the boost applies to non-vector fields such as `importance` and `updated_at`.

## 4. Memory Extraction and Consolidation

This section separates memory extraction and consolidation decisions into explicit classes.

1. `MemoryExtractor`: extracts memory candidates from conversation messages
2. `MemoryConsolidator`: compares a new candidate with nearby existing memories and chooses `ADD` / `UPDATE` / `DELETE` / `NONE`
3. `AzureOpenAIMemoryIntelligence`: a small helper around Azure OpenAI JSON completions and embeddings

Rule-based fallbacks are included so the notebook structure can still be inspected when the LLM configuration is unavailable.

In [ ]:
class AzureOpenAIMemoryIntelligence:
    def __init__(
        self,
        *,
        endpoint: str | None,
        api_key: str | None,
        api_version: str,
        chat_deployment: str | None,
        embedding_deployment: str | None,
        embedding_dims: int,
    ):
        self.endpoint = endpoint
        self.api_key = api_key
        self.api_version = api_version
        self.chat_deployment = chat_deployment
        self.embedding_deployment = embedding_deployment
        self.embedding_dims = embedding_dims
        self.client = None
        if endpoint and (chat_deployment or embedding_deployment):
            try:
                from openai import AzureOpenAI

                kwargs = {"azure_endpoint": endpoint, "api_version": api_version}
                if api_key:
                    kwargs["api_key"] = api_key
                self.client = AzureOpenAI(**kwargs)
            except Exception as error:
                print("Could not initialize AzureOpenAI client. Using fallback:", error)

    @property
    def can_chat(self) -> bool:
        return bool(self.client and self.chat_deployment)

    @property
    def can_embed(self) -> bool:
        return bool(self.client and self.embedding_deployment)

    def complete_json(self, *, system_prompt: str, user_prompt: str) -> dict[str, Any] | None:
        if not self.can_chat:
            return None
        response = self.client.chat.completions.create(
            model=self.chat_deployment,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            response_format={"type": "json_object"},
            temperature=0.1,
        )
        content = response.choices[0].message.content or "{}"
        try:
            return json.loads(content)
        except json.JSONDecodeError:
            match = re.search(r"\{.*\}", content, flags=re.DOTALL)
            return json.loads(match.group(0)) if match else None

    def embed(self, text: str) -> list[float]:
        if not self.can_embed:
            return stable_hash_embedding(text, dimensions=self.embedding_dims)
        response = self.client.embeddings.create(model=self.embedding_deployment, input=text)
        return list(response.data[0].embedding)


def stable_hash_embedding(text: str, dimensions: int = 1536) -> list[float]:
    buckets = [0.0] * dimensions
    tokens = re.findall(r"\w+", text.casefold()) or [text]
    for token in tokens:
        token_hash = hash(token)
        buckets[abs(token_hash) % dimensions] += 1.0 if token_hash >= 0 else -1.0
    norm = math.sqrt(sum(value * value for value in buckets)) or 1.0
    return [value / norm for value in buckets]


def coerce_unit_score(value: Any, *, default: float) -> float:
    if value is None:
        return default

    if isinstance(value, (int, float)):
        score = float(value)
    elif isinstance(value, str):
        text = value.strip().casefold()
        if not text:
            return default
        label_map = {
            "very high": 0.95,
            "high": 0.85,
            "medium": 0.60,
            "mid": 0.60,
            "low": 0.35,
            "very low": 0.15,
        }
        if text in label_map:
            score = label_map[text]
        else:
            match = re.search(r"-?\d+(?:\.\d+)?", text)
            if not match:
                return default
            score = float(match.group(0))
            if "%" in text:
                score /= 100.0
    else:
        return default

    if score > 1.0 and score <= 100.0:
        score /= 100.0
    return max(0.0, min(1.0, score))


class MemoryExtractor:
    def __init__(self, intelligence: AzureOpenAIMemoryIntelligence | None = None):
        self.intelligence = intelligence

    def extract(self, messages: list[dict[str, str]]) -> list[MemoryCandidate]:
        if self.intelligence and self.intelligence.can_chat:
            extracted = self._extract_with_llm(messages)
            if extracted:
                return extracted
        return self._extract_with_rules(messages)

    def _extract_with_llm(self, messages: list[dict[str, str]]) -> list[MemoryCandidate]:
        system_prompt = """
Extract durable memories for the user from the conversation.
Return JSON containing only important memories.
Each memory item must include memory, category, importance, confidence, source, expires_at, and metadata.
Use the user's language. Ignore assistant-only claims. Prefer durable settings, plans, constraints, identity facts, and recurring needs.
Do not store temporary facts that will not be useful later.
""".strip()
        user_prompt = json.dumps({"messages": messages}, ensure_ascii=False)
        payload = self.intelligence.complete_json(system_prompt=system_prompt, user_prompt=user_prompt)
        candidates = []
        for item in (payload or {}).get("memories", []):
            if not isinstance(item, dict) or not item.get("memory"):
                continue
            candidates.append(
                MemoryCandidate(
                    memory=str(item["memory"]),
                    category=str(item.get("category") or "general"),
                    importance=coerce_unit_score(item.get("importance"), default=0.5),
                    confidence=coerce_unit_score(item.get("confidence"), default=0.7),
                    source=str(item.get("source") or "conversation"),
                    expires_at=to_utc_datetime(item.get("expires_at")),
                    metadata=item.get("metadata") if isinstance(item.get("metadata"), dict) else {},
                )
            )
        return candidates

    def _extract_with_rules(self, messages: list[dict[str, str]]) -> list[MemoryCandidate]:
        candidates: list[MemoryCandidate] = []
        durable_patterns = [
            (r"(prefer|like|dislike|always|usually|need|must|my name is|i am|for trips|for work)", "preference"),
            (r"(prefer|like|dislike|always|usually|my name is|i am|for trips|for work)", "preference"),
        ]
        for message in messages:
            if message.get("role") != "user":
                continue
            content = message.get("content", "")
            for sentence in re.split(r"(?<=[.!?])\s+|\n+", content):
                sentence = sentence.strip()
                if not sentence:
                    continue
                category = next((name for pattern, name in durable_patterns if re.search(pattern, sentence, re.IGNORECASE)), None)
                if category:
                    candidates.append(
                        MemoryCandidate(
                            memory=sentence,
                            category=category,
                            importance=0.65,
                            confidence=0.55,
                            source="rule-fallback",
                        )
                    )
        return candidates


class MemoryConsolidator:
    def __init__(
        self,
        intelligence: AzureOpenAIMemoryIntelligence | None = None,
        *,
        update_threshold: float = 0.78,
        none_threshold: float = 0.92,
    ):
        self.intelligence = intelligence
        self.update_threshold = update_threshold
        self.none_threshold = none_threshold

    def decide(self, candidate: MemoryCandidate, neighbors: list[MemorySearchResult]) -> ConsolidationDecision:
        if self.intelligence and self.intelligence.can_chat:
            decision = self._decide_with_llm(candidate, neighbors)
            if decision:
                return decision
        return self._decide_with_rules(candidate, neighbors)

    def _decide_with_llm(
        self,
        candidate: MemoryCandidate,
        neighbors: list[MemorySearchResult],
    ) -> ConsolidationDecision | None:
        system_prompt = """
You are a memory consolidation engine.
Choose exactly one event: ADD, UPDATE, DELETE, NONE.
ADD means saving the candidate as a new memory.
UPDATE means merging the candidate into an existing memory while preserving the target ID.
DELETE means marking an existing memory as deleted because the candidate contradicts it or the user asked to remove it.
NONE means the candidate is already covered.
Return JSON only, with event, target_id, merged_memory, and reason.
""".strip()
        payload = {
            "candidate": asdict(candidate) | {"expires_at": dt_to_json(candidate.expires_at)},
            "neighbors": [
                {
                    "id": result.record.id,
                    "memory": result.record.memory,
                    "category": result.record.category,
                    "score": result.score,
                    "importance": result.record.importance,
                    "updated_at": dt_to_json(result.record.updated_at),
                }
                for result in neighbors
            ],
        }
        result = self.intelligence.complete_json(system_prompt=system_prompt, user_prompt=json.dumps(payload, ensure_ascii=False))
        if not result:
            return None
        try:
            event = MemoryEvent(str(result.get("event")))
        except ValueError:
            return None
        return ConsolidationDecision(
            event=event,
            candidate=candidate,
            target_id=result.get("target_id"),
            merged_memory=result.get("merged_memory") or candidate.memory,
            reason=result.get("reason") or "llm",
        )

    def _decide_with_rules(self, candidate: MemoryCandidate, neighbors: list[MemorySearchResult]) -> ConsolidationDecision:
        if not neighbors:
            return ConsolidationDecision(MemoryEvent.ADD, candidate, reason="no similar memory")
        top = neighbors[0]
        score = top.score or 0.0
        candidate_text = normalize_text(candidate.memory)
        existing_text = normalize_text(top.record.memory)
        if candidate_text == existing_text or score >= self.none_threshold:
            return ConsolidationDecision(MemoryEvent.NONE, candidate, target_id=top.record.id, reason="already covered")
        if self._looks_like_forget_request(candidate.memory):
            return ConsolidationDecision(MemoryEvent.DELETE, candidate, target_id=top.record.id, reason="forget/delete intent")
        if score >= self.update_threshold and candidate.category == top.record.category:
            merged = self._merge_text(top.record.memory, candidate.memory)
            return ConsolidationDecision(MemoryEvent.UPDATE, candidate, target_id=top.record.id, merged_memory=merged, reason="similar category")
        return ConsolidationDecision(MemoryEvent.ADD, candidate, reason="new distinct memory")

    def _looks_like_forget_request(self, text: str) -> bool:
        return bool(re.search(r"(forget|delete|do not remember|remove|discard|cancel)", text, re.IGNORECASE))

    def _merge_text(self, existing: str, new_text: str) -> str:
        if normalize_text(new_text) in normalize_text(existing):
            return existing
        if normalize_text(existing) in normalize_text(new_text):
            return new_text
        return f"{existing} / Added: {new_text}"

print("Defined MemoryExtractor and MemoryConsolidator.")

## 5. Azure AI Search Memory Store

Memories are stored as Azure AI Search documents. `MemorySearchOptions` controls retrieval mode, filters, and ranking policy.

This implementation separates `SearchMode` and `RankingProfile`. `SearchMode` only describes how candidates are retrieved: keyword, vector, hybrid, or semantic. `RankingProfile` describes how an Azure AI Search scoring profile should move candidates up or down. `RankingProfile.IMPORTANT` maps to `memory-priority` and boosts `importance`, `retention_score`, and `access_count`. `RankingProfile.RECENT` maps to `memory-recency` and boosts recently updated memories.

| Option | Store behavior |
|---|---|
| `mode=SearchMode.KEYWORD` | Search memories with keyword search only |
| `mode=SearchMode.VECTOR` | Search by embedding vector similarity only |
| `mode=SearchMode.HYBRID` | Combine keyword search and vector search |
| `mode=SearchMode.SEMANTIC` | Add semantic ranking on top of hybrid retrieval |
| `ranking_profile=RankingProfile.NONE` | Do not use an Azure AI Search scoring profile |
| `ranking_profile=RankingProfile.IMPORTANT` | Apply `memory-priority` to boost importance, retention, and access count |
| `ranking_profile=RankingProfile.RECENT` | Apply `memory-recency` to boost recently updated memories |
| `category`, `min_importance`, `min_retention_score` | Convert to Azure AI Search filter expressions |
| `include_archived` | Makes archived memories visible to the search |
| `order_by` | Use only when explicit sorting is more important than relevance score |

Top-level fields such as `created_at`, `updated_at`, `importance`, `retention_score`, and `access_count` can be used directly by Azure AI Search filters, sorts, and scoring functions. The point is not to rerank everything in Python, but to tell the search engine which memory signals matter.

### Why `RankingProfile.IMPORTANT` and `RankingProfile.RECENT` Matter

Retrieval mode handles similarity. Ranking profile handles memory lifecycle. This separation lets you combine `SearchMode.VECTOR` or `SearchMode.HYBRID` with importance-oriented or recency-oriented boosting.

| Desired behavior | Why keyword/vector alone is not enough |
|---|---|
| Prefer recently changed plans | Similar but old conditions can still match strongly |
| Prefer durable user preferences | Important old memories can lose to newer noise |
| Prefer frequently recalled constraints | Vector similarity does not know what is frequently used |
| Weaken temporary plans naturally | Semantic similarity alone does not know expiration or retention value |

Azure AI Search scoring profiles can be specified for vector and hybrid queries. The boost does not apply to the vector value itself; it applies to non-vector fields such as `importance` and `updated_at` within the candidate set returned by retrieval.

In [ ]:
# pyright: reportMissingImports=false

from datetime import timedelta

from azure.core.credentials import AzureKeyCredential
from azure.identity import DefaultAzureCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    FreshnessScoringFunction,
    FreshnessScoringParameters,
    HnswAlgorithmConfiguration,
    MagnitudeScoringFunction,
    MagnitudeScoringParameters,
    ScoringProfile,
    SearchableField,
    SearchField,
    SearchFieldDataType,
    SearchIndex,
    SemanticConfiguration,
    SemanticField,
    SemanticPrioritizedFields,
    SemanticSearch,
    SimpleField,
    TextWeights,
    VectorSearch,
    VectorSearchProfile,
)
from azure.search.documents.models import QueryType, VectorizedQuery


def create_vectorized_query(*, vector: list[float], k: int, fields: str) -> VectorizedQuery:
    attribute_map = getattr(VectorizedQuery, "_attribute_map", {}) or {}
    supported_attributes = set(attribute_map)
    kwargs: dict[str, Any] = {"vector": vector, "fields": fields}
    if "k_nearest_neighbors" in supported_attributes:
        kwargs["k_nearest_neighbors"] = k
    else:
        kwargs["k"] = k
    return VectorizedQuery(**kwargs)


class AzureAISearchMemoryStore:
    def __init__(
        self,
        *,
        endpoint: str,
        index_name: str,
        api_key: str | None,
        embedding_dims: int,
        semantic_configuration_name: str = "memory-semantic",
        vector_field_name: str = "embedding",
        auto_create: bool = True,
    ):
        self.endpoint = endpoint
        self.index_name = index_name
        self.embedding_dims = embedding_dims
        self.semantic_configuration_name = semantic_configuration_name
        self.vector_field_name = vector_field_name
        credential = AzureKeyCredential(api_key) if api_key else DefaultAzureCredential()
        self.search_client = SearchClient(endpoint=endpoint, index_name=index_name, credential=credential)
        self.index_client = SearchIndexClient(endpoint=endpoint, credential=credential)
        if auto_create:
            self.ensure_index()

    def ensure_index(self) -> None:
        existing_names = set(self.index_client.list_index_names())
        if self.index_name in existing_names:
            self._ensure_scoring_profiles()
            return

        fields = [
            SimpleField(name="id", type=SearchFieldDataType.String, key=True),
            SimpleField(name="user_id", type=SearchFieldDataType.String, filterable=True, sortable=True),
            SimpleField(name="agent_id", type=SearchFieldDataType.String, filterable=True, sortable=True),
            SearchableField(name="memory", type=SearchFieldDataType.String, searchable=True),
            SearchableField(name="summary", type=SearchFieldDataType.String, searchable=True),
            SearchableField(name="category", type=SearchFieldDataType.String, searchable=True, filterable=True, facetable=True),
            SimpleField(name="status", type=SearchFieldDataType.String, filterable=True, facetable=True),
            SimpleField(name="source", type=SearchFieldDataType.String, filterable=True, facetable=True),
            SimpleField(name="importance", type=SearchFieldDataType.Double, filterable=True, sortable=True, facetable=True),
            SimpleField(name="confidence", type=SearchFieldDataType.Double, filterable=True, sortable=True),
            SimpleField(name="retention_score", type=SearchFieldDataType.Double, filterable=True, sortable=True),
            SimpleField(name="access_count", type=SearchFieldDataType.Int32, filterable=True, sortable=True),
            SimpleField(name="created_at", type=SearchFieldDataType.DateTimeOffset, filterable=True, sortable=True),
            SimpleField(name="updated_at", type=SearchFieldDataType.DateTimeOffset, filterable=True, sortable=True),
            SimpleField(name="last_accessed_at", type=SearchFieldDataType.DateTimeOffset, filterable=True, sortable=True),
            SimpleField(name="expires_at", type=SearchFieldDataType.DateTimeOffset, filterable=True, sortable=True),
            SimpleField(name="metadata_json", type=SearchFieldDataType.String),
            SearchField(
                name=self.vector_field_name,
                type="Collection(Edm.Single)",
                searchable=True,
                vector_search_dimensions=self.embedding_dims,
                vector_search_profile_name="memory-vector-profile",
            ),
        ]
        vector_search = VectorSearch(
            profiles=[VectorSearchProfile(name="memory-vector-profile", algorithm_configuration_name="memory-hnsw")],
            algorithms=[HnswAlgorithmConfiguration(name="memory-hnsw")],
        )
        semantic_search = SemanticSearch(
            configurations=[
                SemanticConfiguration(
                    name=self.semantic_configuration_name,
                    prioritized_fields=SemanticPrioritizedFields(
                        content_fields=[SemanticField(field_name="memory"), SemanticField(field_name="summary")],
                        keywords_fields=[SemanticField(field_name="category")],
                    ),
                )
            ]
        )
        index = SearchIndex(
            name=self.index_name,
            fields=fields,
            vector_search=vector_search,
            semantic_search=semantic_search,
            scoring_profiles=self._memory_scoring_profiles(),
        )
        self.index_client.create_or_update_index(index)

    def close(self) -> None:
        self.search_client.close()
        self.index_client.close()

    def upsert(self, record: MemoryRecord, embedding: list[float]) -> None:
        response = self.search_client.merge_or_upload_documents([record.to_search_document(embedding=embedding)])
        self._raise_if_failed(response, "upsert", record.id)

    def mark_status(self, memory_id: str, status: MemoryStatus) -> None:
        response = self.search_client.merge_documents([{"id": memory_id, "status": str(status), "updated_at": utc_now()}])
        self._raise_if_failed(response, "mark_status", memory_id)

    def get(self, memory_id: str) -> MemoryRecord | None:
        try:
            document = self.search_client.get_document(key=memory_id)
        except Exception:
            return None
        return MemoryRecord.from_search_document(dict(document))

    def search(
        self,
        *,
        query: str,
        query_vector: list[float] | None,
        user_id: str,
        agent_id: str,
        options: MemorySearchOptions,
    ) -> list[MemorySearchResult]:
        filter_expression = self._filter_expression(user_id=user_id, agent_id=agent_id, options=options)
        order_by = self._order_by_for(options)
        scoring_profile = self._scoring_profile_for(options)
        params: dict[str, Any] = {
            "filter": filter_expression,
            "top": options.top,
            "select": [
                "id", "user_id", "agent_id", "memory", "summary", "category", "status", "source",
                "importance", "confidence", "retention_score", "access_count", "created_at", "updated_at",
                "last_accessed_at", "expires_at", "metadata_json",
            ],
        }

        mode = options.mode
        if mode == SearchMode.VECTOR:
            if query_vector is None:
                return []
            params["vector_queries"] = [create_vectorized_query(vector=query_vector, k=options.top, fields=self.vector_field_name)]
        elif mode in {SearchMode.HYBRID, SearchMode.SEMANTIC}:
            params["search_text"] = query or "*"
            if query_vector is not None:
                params["vector_queries"] = [create_vectorized_query(vector=query_vector, k=max(options.top, 20), fields=self.vector_field_name)]
            params["search_fields"] = ["memory", "summary", "category"]
        else:
            params["search_text"] = query or "*"
            params["search_fields"] = ["memory", "summary", "category"]

        if mode == SearchMode.SEMANTIC:
            params["query_type"] = QueryType.SEMANTIC
            params["semantic_configuration_name"] = options.semantic_configuration_name
        if scoring_profile:
            params["scoring_profile"] = scoring_profile
        if order_by:
            params["order_by"] = [order_by]

        search_results = self.search_client.search(**params)
        results = []
        for document in search_results:
            record = MemoryRecord.from_search_document(dict(document))
            score = document.get("@search.score")
            reranker_score = document.get("@search.reranker_score")
            results.append(MemorySearchResult(record=record, score=score, reranker_score=reranker_score))
        return results

    def touch_results(self, results: list[MemorySearchResult], policy: HumanMemoryPolicy) -> None:
        documents = []
        for result in results:
            record = policy.touch(result.record)
            documents.append({
                "id": record.id,
                "last_accessed_at": record.last_accessed_at,
                "access_count": record.access_count,
                "retention_score": record.retention_score,
                "status": str(record.status),
            })
        if documents:
            response = self.search_client.merge_documents(documents)
            self._raise_if_failed(response, "touch_results", "batch")

    def apply_forgetting(self, policy: HumanMemoryPolicy, *, limit: int = 100, dry_run: bool = True) -> list[dict[str, Any]]:
        results = self.search_client.search(
            search_text="*",
            filter="status ne 'deleted'",
            top=limit,
            order_by=["retention_score asc"],
        )
        updates = []
        for document in results:
            record = MemoryRecord.from_search_document(dict(document))
            new_score = policy.retention_score(record)
            new_status = policy.status_for(record)
            updates.append({"id": record.id, "memory": record.memory, "retention_score": new_score, "status": str(new_status)})
        if not dry_run and updates:
            response = self.search_client.merge_documents([
                {"id": item["id"], "retention_score": item["retention_score"], "status": item["status"]}
                for item in updates
            ])
            self._raise_if_failed(response, "apply_forgetting", "batch")
        return updates

    def _memory_scoring_profiles(self) -> list[ScoringProfile]:
        text_weights = TextWeights(weights={"memory": 3.0, "summary": 2.0, "category": 1.2})
        priority_functions = [
            MagnitudeScoringFunction(
                field_name="importance",
                boost=8.0,
                parameters=MagnitudeScoringParameters(
                    boosting_range_start=0.0,
                    boosting_range_end=1.0,
                    should_boost_beyond_range_by_constant=True,
                ),
                interpolation="linear",
            ),
            MagnitudeScoringFunction(
                field_name="retention_score",
                boost=4.0,
                parameters=MagnitudeScoringParameters(
                    boosting_range_start=0.0,
                    boosting_range_end=1.0,
                    should_boost_beyond_range_by_constant=True,
                ),
                interpolation="linear",
            ),
            MagnitudeScoringFunction(
                field_name="access_count",
                boost=4.0,
                parameters=MagnitudeScoringParameters(
                    boosting_range_start=0.0,
                    boosting_range_end=20.0,
                    should_boost_beyond_range_by_constant=True,
                ),
                interpolation="logarithmic",
            ),
        ]
        recency_functions = [
            FreshnessScoringFunction(
                field_name="updated_at",
                boost=8.0,
                parameters=FreshnessScoringParameters(boosting_duration=timedelta(days=90)),
                interpolation="linear",
            ),
        ]
        return [
            ScoringProfile(
                name=MEMORY_PRIORITY_SCORING_PROFILE,
                text_weights=text_weights,
                functions=priority_functions,
                function_aggregation="sum",
            ),
            ScoringProfile(
                name=MEMORY_RECENCY_SCORING_PROFILE,
                text_weights=text_weights,
                functions=recency_functions,
                function_aggregation="sum",
            ),
        ]

    def _ensure_scoring_profiles(self) -> None:
        index = self.index_client.get_index(self.index_name)
        desired_profiles = {profile.name: profile for profile in self._memory_scoring_profiles()}
        existing_profiles = [
            profile for profile in (index.scoring_profiles or [])
            if profile.name not in desired_profiles
        ]
        index.scoring_profiles = existing_profiles + list(desired_profiles.values())
        index.default_scoring_profile = None
        self.index_client.create_or_update_index(index)

    def _scoring_profile_for(self, options: MemorySearchOptions) -> str | None:
        if options.ranking_profile == RankingProfile.IMPORTANT:
            return MEMORY_PRIORITY_SCORING_PROFILE
        if options.ranking_profile == RankingProfile.RECENT:
            return MEMORY_RECENCY_SCORING_PROFILE
        return None

    def _order_by_for(self, options: MemorySearchOptions) -> str | None:
        return options.order_by

    def _filter_expression(self, *, user_id: str, agent_id: str, options: MemorySearchOptions) -> str:
        parts = [
            f"user_id eq '{self._odata_quote(user_id)}'",
            f"agent_id eq '{self._odata_quote(agent_id)}'",
        ]
        statuses = [MemoryStatus.ACTIVE]
        if options.include_archived:
            statuses.append(MemoryStatus.ARCHIVED)
        status_csv = ",".join(str(status) for status in statuses)
        parts.append(f"search.in(status, '{status_csv}', ',')")
        if options.category:
            parts.append(f"category eq '{self._odata_quote(options.category)}'")
        if options.min_importance is not None:
            parts.append(f"importance ge {options.min_importance}")
        if options.min_retention_score is not None:
            parts.append(f"retention_score ge {options.min_retention_score}")
        return " and ".join(parts)

    def _odata_quote(self, value: str) -> str:
        return value.replace("'", "''")

    def _raise_if_failed(self, response: list[Any], operation: str, target: str) -> None:
        failed = [item for item in response if not getattr(item, "succeeded", False)]
        if failed:
            details = ", ".join(str(getattr(item, "error_message", "unknown")) for item in failed)
            raise RuntimeError(f"{operation} failed for {target}: {details}")

print("Defined AzureAISearchMemoryStore.")

## 6. Custom `ContextProvider`

This section connects the memory components to the Agent Framework execution cycle. `ContextProvider` lets us inject behavior before and after agent execution.

When the agent runs, memory processing follows this flow:

| Timing | Trigger | What this notebook does | Effect on scores and forgetting |
|---|---|---|---|
| Before agent execution | `before_run()` | Search memories related to the user input and add them to context | Search with `search_options.mode` and `search_options.ranking_profile`, then touch only the memories actually used |
| During model call | Model invocation | Generate the answer with the injected memories | The memory store is not updated at this moment |
| After agent execution | `after_run()` | Extract memory candidates from the user input and response, then decide add/update/delete/none | Added or updated memories get a fresh `retention_score` |
| Post-run check | `apply_forgetting_after_run=True` | Run `apply_forgetting(..., dry_run=True)` to preview weak memories | This notebook previews changes only; it does not write forgetting results back by default |

The important point is that the notebook does not recompute every memory on every agent call. Before execution, only retrieved memories are touched. After execution, only newly added or consolidated memories are updated.

Forgetting is also preview-only by default. With `dry_run=True`, the notebook calculates expected `retention_score` and `status`, but does not write them back to Azure AI Search. In production, `apply_forgetting(..., dry_run=False)` should be run as a scheduled or low-frequency maintenance job.

`search_options` controls the default memory search used by `before_run()`. The default uses `SearchMode.HYBRID`, `RankingProfile.IMPORTANT`, `top=5`, and `min_retention_score=0.1`. To emphasize recently updated conditions, keep the same retrieval mode and set `ranking_profile=RankingProfile.RECENT`.

In [ ]:
# pyright: reportMissingImports=false, reportUnusedImport=false

try:
    from agent_framework import Message
    from agent_framework._sessions import ContextProvider
except Exception:
    @dataclass
    class Message:
        role: str
        contents: list[str]
        message_id: str | None = None
        author_name: str | None = None
        additional_properties: dict[str, Any] = field(default_factory=dict)

        @property
        def text(self) -> str:
            return "\n".join(str(item) for item in self.contents)

    class ContextProvider:
        def __init__(self, source_id: str):
            self.source_id = source_id


class AzureAISearchMemoryContextProvider(ContextProvider):
    DEFAULT_CONTEXT_PROMPT = "## Relevant memories\nUse these memories only when they help answer the user."

    def __init__(
        self,
        *,
        store: AzureAISearchMemoryStore,
        intelligence: AzureOpenAIMemoryIntelligence,
        extractor: MemoryExtractor,
        consolidator: MemoryConsolidator,
        policy: HumanMemoryPolicy,
        user_id: str,
        agent_id: str,
        source_id: str = "custom_aisearch_memory",
        search_options: MemorySearchOptions | None = None,
        context_prompt: str | None = None,
        apply_forgetting_after_run: bool = True,
        log_search_results: bool = True,
        log_empty_search_diagnostics: bool = True,
    ):
        super().__init__(source_id)
        self.store = store
        self.intelligence = intelligence
        self.extractor = extractor
        self.consolidator = consolidator
        self.policy = policy
        self.user_id = user_id
        self.agent_id = agent_id
        self.search_options = search_options or MemorySearchOptions(mode=SearchMode.HYBRID, top=5, min_retention_score=0.1)
        self.context_prompt = context_prompt or self.DEFAULT_CONTEXT_PROMPT
        self.apply_forgetting_after_run = apply_forgetting_after_run
        self.log_search_results = log_search_results
        self.log_empty_search_diagnostics = log_empty_search_diagnostics

    async def before_run(
        self,
        *,
        agent: Any,
        session: Any,
        context: Any,
        state: dict[str, Any],
    ) -> None:
        _ = (agent, session)
        query = "\n".join(message.text for message in context.input_messages if getattr(message, "text", "").strip())
        if not query.strip():
            return
        query_vector = self.intelligence.embed(query)
        results = self.store.search(
            query=query,
            query_vector=query_vector,
            user_id=self.user_id,
            agent_id=self.agent_id,
            options=self.search_options,
        )
        self.store.touch_results(results, self.policy)
        state["last_memory_search"] = [self._search_result_to_state(result) for result in results]
        self._log_automatic_search(query, results)
        if results:
            context.extend_messages(self, [Message(role="user", contents=[self._format_context(results)])])

    async def after_run(
        self,
        *,
        agent: Any,
        session: Any,
        context: Any,
        state: dict[str, Any],
    ) -> None:
        _ = (agent, session)
        messages = [self._message_to_dict(message) for message in context.input_messages]
        if context.response and context.response.messages:
            messages.extend(self._message_to_dict(message) for message in context.response.messages)
        decisions = await self.ingest_messages(messages, session_id=context.session_id)
        state["last_memory_decisions"] = [self._decision_to_state(decision) for decision in decisions]
        if self.apply_forgetting_after_run:
            state["forgetting_preview"] = self.store.apply_forgetting(self.policy, dry_run=True, limit=25)

    async def ingest_messages(self, messages: list[dict[str, str]], *, session_id: str | None = None) -> list[ConsolidationDecision]:
        candidates = self.extractor.extract(messages)
        decisions: list[ConsolidationDecision] = []
        for candidate in candidates:
            candidate_vector = self.intelligence.embed(candidate.memory)
            neighbors = self.store.search(
                query=candidate.memory,
                query_vector=candidate_vector,
                user_id=self.user_id,
                agent_id=self.agent_id,
                options=MemorySearchOptions(mode=SearchMode.HYBRID, top=5, include_archived=True),
            )
            decision = self.consolidator.decide(candidate, neighbors)
            self._apply_decision(decision, candidate_vector, session_id=session_id)
            decisions.append(decision)
        return decisions

    def search_memories(self, query: str, options: MemorySearchOptions | None = None) -> list[MemorySearchResult]:
        options = options or self.search_options
        return self.store.search(
            query=query,
            query_vector=self.intelligence.embed(query),
            user_id=self.user_id,
            agent_id=self.agent_id,
            options=options,
        )

    def _apply_decision(
        self,
        decision: ConsolidationDecision,
        candidate_vector: list[float],
        *,
        session_id: str | None,
    ) -> None:
        now = utc_now()
        if decision.event == MemoryEvent.NONE:
            return
        if decision.event == MemoryEvent.DELETE and decision.target_id:
            self.store.mark_status(decision.target_id, MemoryStatus.DELETED)
            return
        if decision.event == MemoryEvent.UPDATE and decision.target_id:
            existing = self.store.get(decision.target_id)
            if not existing:
                return
            existing.memory = decision.merged_memory or decision.candidate.memory
            existing.summary = existing.memory
            existing.category = decision.candidate.category or existing.category
            existing.importance = max(existing.importance, decision.candidate.importance)
            existing.confidence = max(existing.confidence, decision.candidate.confidence)
            existing.updated_at = now
            existing.expires_at = decision.candidate.expires_at or existing.expires_at
            existing.metadata.update(decision.candidate.metadata)
            existing.metadata["last_consolidation_reason"] = decision.reason
            existing.retention_score = self.policy.retention_score(existing, now=now)
            self.store.upsert(existing, embedding=self.intelligence.embed(existing.memory))
            return
        if decision.event == MemoryEvent.ADD:
            record = MemoryRecord(
                id=str(uuid.uuid4()),
                user_id=self.user_id,
                agent_id=self.agent_id,
                memory=decision.candidate.memory,
                summary=decision.candidate.memory,
                category=decision.candidate.category,
                importance=decision.candidate.importance,
                confidence=decision.candidate.confidence,
                source=decision.candidate.source,
                created_at=now,
                updated_at=now,
                expires_at=decision.candidate.expires_at,
                metadata={**decision.candidate.metadata, "session_id": session_id, "consolidation_reason": decision.reason},
            )
            record.retention_score = self.policy.retention_score(record, now=now)
            self.store.upsert(record, embedding=candidate_vector)

    def _format_context(self, results: list[MemorySearchResult]) -> str:
        lines = [self.context_prompt]
        for index, result in enumerate(results, start=1):
            record = result.record
            lines.append(
                f"{index}. [{record.category}; importance={record.importance:.2f}; retention={record.retention_score:.2f}] "
                f"{record.memory}"
            )
        return "\n".join(lines)

    def _log_automatic_search(self, query: str, results: list[MemorySearchResult]) -> None:
        if not self.log_search_results:
            return
        filter_expression = self.store._filter_expression(
            user_id=self.user_id,
            agent_id=self.agent_id,
            options=self.search_options,
        )
        payload = {
            "source_id": self.source_id,
            "user_id": self.user_id,
            "agent_id": self.agent_id,
            "query": query,
            "mode": str(self.search_options.mode),
            "top": self.search_options.top,
            "filter": filter_expression,
            "min_retention_score": self.search_options.min_retention_score,
            "include_archived": self.search_options.include_archived,
            "result_count": len(results),
            "results": [self._search_result_to_log_item(index, result) for index, result in enumerate(results, start=1)],
        }
        if not results and self.log_empty_search_diagnostics:
            payload["empty_result_diagnostics"] = self._empty_search_diagnostics(query)
        custom_memory_logger.info("Automatic memory search results:\n%s", json.dumps(payload, ensure_ascii=False, indent=2))

    def _search_result_to_log_item(self, index: int, result: MemorySearchResult) -> dict[str, Any]:
        record = result.record
        return {
            "rank": index,
            "id": record.id,
            "score": result.score,
            "reranker_score": result.reranker_score,
            "memory": record.memory,
            "category": record.category,
            "status": str(record.status),
            "importance": record.importance,
            "retention_score": record.retention_score,
            "access_count": record.access_count,
            "updated_at": dt_to_json(record.updated_at),
            "last_accessed_at": dt_to_json(record.last_accessed_at),
        }

    def _empty_search_diagnostics(self, query: str) -> dict[str, Any]:
        diagnostics: dict[str, Any] = {}
        try:
            relaxed_options = MemorySearchOptions(
                mode=SearchMode.KEYWORD,
                top=5,
                category=self.search_options.category,
                include_archived=True,
            )
            relaxed_results = self.store.search(
                query=query,
                query_vector=None,
                user_id=self.user_id,
                agent_id=self.agent_id,
                options=relaxed_options,
            )
            any_memory_results = self.store.search(
                query="*",
                query_vector=None,
                user_id=self.user_id,
                agent_id=self.agent_id,
                options=relaxed_options,
            )
            diagnostics["relaxed_filter"] = self.store._filter_expression(
                user_id=self.user_id,
                agent_id=self.agent_id,
                options=relaxed_options,
            )
            diagnostics["same_query_without_retention_filter_count"] = len(relaxed_results)
            diagnostics["same_query_without_retention_filter_results"] = [
                self._search_result_to_log_item(index, result)
                for index, result in enumerate(relaxed_results, start=1)
            ]
            diagnostics["visible_memory_sample_count"] = len(any_memory_results)
            diagnostics["visible_memory_sample"] = [
                self._search_result_to_log_item(index, result)
                for index, result in enumerate(any_memory_results, start=1)
            ]
        except Exception as error:
            diagnostics["error"] = repr(error)
        return diagnostics

    def _message_to_dict(self, message: Any) -> dict[str, str]:
        role = str(getattr(message, "role", "user"))
        text = str(getattr(message, "text", "") or "")
        return {"role": role, "content": text}

    def _search_result_to_state(self, result: MemorySearchResult) -> dict[str, Any]:
        return {
            "id": result.record.id,
            "memory": result.record.memory,
            "category": result.record.category,
            "score": result.score,
            "importance": result.record.importance,
            "retention_score": result.record.retention_score,
        }

    def _decision_to_state(self, decision: ConsolidationDecision) -> dict[str, Any]:
        return {
            "event": str(decision.event),
            "target_id": decision.target_id,
            "memory": decision.merged_memory or decision.candidate.memory,
            "reason": decision.reason,
        }

print("Defined AzureAISearchMemoryContextProvider.")

## 7. Initialize the Memory Provider

This cell combines the extractor, consolidator, forgetting policy, and Azure AI Search store into `custom_memory_provider`. Later cells use this provider to add, search, and forget memories.

The initialization uses `MemorySearchOptions(mode=SearchMode.HYBRID, top=5, min_retention_score=0.1)` as the default pre-run search. It retrieves relevant memories with hybrid search, injects at most five memories into context, and filters out memories with low retention scores.

Set `CUSTOM_MEMORY_RESET_INDEX=true` to delete and recreate the test index. Existing indexes are updated with scoring profiles when possible, but recreating the index makes schema-level changes easier to verify.

In [ ]:
# pyright: reportMissingImports=false

memory_intelligence = AzureOpenAIMemoryIntelligence(
    endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION,
    chat_deployment=AZURE_OPENAI_CHAT_DEPLOYMENT,
    embedding_deployment=AZURE_OPENAI_EMBEDDING_DEPLOYMENT,
    embedding_dims=AZURE_OPENAI_EMBEDDING_DIMS,
)
memory_extractor = MemoryExtractor(memory_intelligence)
memory_policy = HumanMemoryPolicy(half_life_days=45, access_boost=0.04)
memory_consolidator = MemoryConsolidator(memory_intelligence)

memory_store = None
custom_memory_provider = None

if not SEARCH_READY:
    print("Skipped: Azure AI Search endpoint or index name is missing.")
else:
    if os.getenv("CUSTOM_MEMORY_RESET_INDEX", "false").casefold() in {"1", "true", "yes", "y"}:
        credential = AzureKeyCredential(AZURE_SEARCH_API_KEY) if AZURE_SEARCH_API_KEY else DefaultAzureCredential()
        reset_client = SearchIndexClient(endpoint=AZURE_SEARCH_ENDPOINT, credential=credential)
        if AZURE_SEARCH_INDEX_NAME in set(reset_client.list_index_names()):
            reset_client.delete_index(AZURE_SEARCH_INDEX_NAME)
            print("Deleted existing index:", AZURE_SEARCH_INDEX_NAME)
        reset_client.close()

    memory_store = AzureAISearchMemoryStore(
        endpoint=AZURE_SEARCH_ENDPOINT,
        index_name=AZURE_SEARCH_INDEX_NAME,
        api_key=AZURE_SEARCH_API_KEY,
        embedding_dims=AZURE_OPENAI_EMBEDDING_DIMS,
    )
    custom_memory_provider = AzureAISearchMemoryContextProvider(
        store=memory_store,
        intelligence=memory_intelligence,
        extractor=memory_extractor,
        consolidator=memory_consolidator,
        policy=memory_policy,
        user_id=DEMO_USER_ID,
        agent_id=DEMO_AGENT_ID,
        search_options=MemorySearchOptions(mode=SearchMode.HYBRID, top=5, min_retention_score=0.1),
    )
    print("Initialized custom memory provider.")
    print("  index:", AZURE_SEARCH_INDEX_NAME)
    print("  user_id:", DEMO_USER_ID)
    print("  agent_id:", DEMO_AGENT_ID)
    print("  chat_llm:", memory_intelligence.can_chat)
    print("  embedding:", memory_intelligence.can_embed or "stable_hash_fallback")

## 8. Add Memories and Compare Retrieval Modes

This cell calls `ingest_messages()` directly. In a real app, `after_run()` would call the same flow after an agent response.

The second half compares `SearchMode` and `RankingProfile` combinations for the same question. The point is not that one mode is always correct, but that candidate retrieval and ranking policy are separate axes. Callers specify `RankingProfile`, not raw Azure AI Search scoring profile names.

| Retrieval mode | What it emphasizes | Good fit |
|---|---|---|
| `SearchMode.KEYWORD` | Exact term matches in memory text | Proper nouns, product names, explicit keywords |
| `SearchMode.VECTOR` | Semantic similarity | Paraphrases and vague questions |
| `SearchMode.HYBRID` | Keyword + vector retrieval | Default memory retrieval for agents |
| `SearchMode.SEMANTIC` | Semantic ranking on top of hybrid retrieval | Higher-quality result ordering and captions |

| Ranking policy | Internal Azure AI Search profile | Good fit |
|---|---|---|
| `RankingProfile.NONE` | None | Compare raw keyword/vector/hybrid behavior |
| `RankingProfile.IMPORTANT` | `memory-priority` | Durable preferences and strong constraints |
| `RankingProfile.RECENT` | `memory-recency` | Recent plans and newly added conditions |

This cell first uses `RankingProfile.NONE` to compare retrieval modes. It then combines the same `SearchMode.HYBRID` with `RankingProfile.IMPORTANT` and `RankingProfile.RECENT` to show how scoring profiles change ranking.

In [ ]:
demo_messages = [
    {"role": "user", "content": "My name is Haruka. For weekend trips to Osaka, I prioritize hotels near the station and aisle seats."},
    {"role": "assistant", "content": "For Osaka trips, I will prioritize hotels near the station and aisle seats."},
    {"role": "user", "content": "For meals, I want restaurants with vegetarian options. Old plans can be forgotten after next month."},
]

if custom_memory_provider is None:
    print("Skipped: custom_memory_provider is not initialized.")
else:
    decisions = await custom_memory_provider.ingest_messages(demo_messages, session_id="notebook-demo")
    print("Consolidation decisions:")
    print(json.dumps([custom_memory_provider._decision_to_state(decision) for decision in decisions], ensure_ascii=False, indent=2))

    query = "Osaka trip hotel and dining preferences"
    comparisons = [
        ("KEYWORD", MemorySearchOptions(mode=SearchMode.KEYWORD, ranking_profile=RankingProfile.NONE, top=5, include_archived=True)),
        ("VECTOR", MemorySearchOptions(mode=SearchMode.VECTOR, ranking_profile=RankingProfile.NONE, top=5, include_archived=True)),
        ("HYBRID", MemorySearchOptions(mode=SearchMode.HYBRID, ranking_profile=RankingProfile.NONE, top=5, include_archived=True)),
        ("SEMANTIC", MemorySearchOptions(mode=SearchMode.SEMANTIC, ranking_profile=RankingProfile.NONE, top=5, include_archived=True)),
        ("HYBRID + IMPORTANT", MemorySearchOptions(mode=SearchMode.HYBRID, ranking_profile=RankingProfile.IMPORTANT, top=5, include_archived=True)),
        ("HYBRID + RECENT", MemorySearchOptions(mode=SearchMode.HYBRID, ranking_profile=RankingProfile.RECENT, top=5, include_archived=True)),
    ]
    for label, options in comparisons:
        results = custom_memory_provider.search_memories(query, options=options)
        print(f"\n--- {label} ---")
        print(json.dumps([
            {
                "score": result.score,
                "reranker_score": result.reranker_score,
                "mode": str(options.mode),
                "ranking_profile": str(options.ranking_profile),
                "memory": result.record.memory,
                "category": result.record.category,
                "importance": result.record.importance,
                "retention_score": result.record.retention_score,
                "created_at": dt_to_json(result.record.created_at),
                "updated_at": dt_to_json(result.record.updated_at),
            }
            for result in results
        ], ensure_ascii=False, indent=2))

## 9. Human-Like Memory: Recall, Decay, and Forgetting

This section checks how stored memories can change weight over time. Agent memory becomes useful as it grows, but old plans and weak preferences can hurt answer quality if they are always injected. The policy therefore makes recalled memories easier to keep and unused memories gradually less visible.

The following scenario also shows how `RankingProfile` changes Azure AI Search ranking. `RankingProfile.IMPORTANT` boosts importance, retention, and access count. `RankingProfile.RECENT` boosts recently updated memories. Retrieval mode is still selected with `SearchMode`.

| Feature | Implementation | Effect on search results |
|---|---|---|
| Recall reinforcement | Update `access_count` and `last_accessed_at` for retrieved memories | Frequently recalled memories rank higher with `RankingProfile.IMPORTANT` |
| IMPORTANT vs RECENT | Compare an old important memory with a new lightweight memory | The same retrieval mode can rank candidates differently depending on the ranking profile |
| Time decay | Lower `retention_score` based on elapsed days and half-life | Weak old memories rank lower and can move toward `archived` |
| Old memory visibility | Transition from `active` to `archived` | Default filters hide them, but audit searches can include them |
| Explicit deletion | Mark user-forgotten or contradictory memories as `deleted` | Deleted memories are filtered out even if their score would be high |

The scenario cells below create dedicated test data and verify:

1. Create memories and helper functions for score inspection
2. Search with `RankingProfile.IMPORTANT` and compare ranking before and after recall
3. Compare `RankingProfile.IMPORTANT` and `RankingProfile.RECENT`
4. Mark one memory as `deleted` and confirm filtering, not scoring, removes it
5. Compare forgetting candidates with answer-time search results

### 9-0b. Initialize Scenario Data

This cell creates 12 test memories used by the later checks in section 9. The memories intentionally contain similar terms such as "Osaka trip", "hotel", "near the station", "luggage", "event", and "plan" so ranking differences are easy to inspect.

`scenario_ctx` stores the category name and document IDs used by later cells. Search results include a `role` value so you can see which memory is the target and which ones are merely similar.

The pair below is used to compare `IMPORTANT` and `RECENT` ranking behavior:

- `IMPORTANT_TARGET`: the same query text, old but high importance and frequently accessed
- `RECENT_TARGET`: the same query text, new but low importance

In [ ]:
def delete_all_memory_documents(store: AzureAISearchMemoryStore, *, batch_size: int = 1000) -> int:
    document_ids: list[str] = []
    seen_ids: set[str] = set()
    for document in store.search_client.search(search_text="*", select=["id"]):
        document_id = str(document["id"])
        if document_id not in seen_ids:
            seen_ids.add(document_id)
            document_ids.append(document_id)

    for start in range(0, len(document_ids), batch_size):
        batch_ids = document_ids[start:start + batch_size]
        response = store.search_client.delete_documents(documents=[{"id": document_id} for document_id in batch_ids])
        store._raise_if_failed(response, "delete_documents", "batch")

    return len(document_ids)


scenario_ctx = {}

if memory_store is None:
    print("Skipped: memory_store is not initialized.")
else:
    deleted_count = delete_all_memory_documents(memory_store)
    print(f"Deleted documents from the target index: {deleted_count}")
    print("Cleared scenario state.")

In [ ]:
import time


def summarize_scored_results(results: list[MemorySearchResult]) -> list[dict[str, Any]]:
    return [
        {
            "rank": index,
            "id": result.record.id,
            "role": result.record.metadata.get("scenario_role"),
            "score": result.score,
            "reranker_score": result.reranker_score,
            "memory": result.record.memory,
            "importance": result.record.importance,
            "retention_score": result.record.retention_score,
            "access_count": result.record.access_count,
            "status": str(result.record.status),
            "updated_at": dt_to_json(result.record.updated_at),
            "last_accessed_at": dt_to_json(result.record.last_accessed_at),
        }
        for index, result in enumerate(results, start=1)
    ]


def wait_for_scenario_documents(store: AzureAISearchMemoryStore, *, category: str, expected_count: int, timeout_seconds: float = 15.0) -> int:
    deadline = time.monotonic() + timeout_seconds
    observed_count = 0
    while time.monotonic() < deadline:
        documents = list(
            store.search_client.search(
                search_text="*",
                filter=f"category eq '{store._odata_quote(category)}'",
                select=["id"],
                top=expected_count,
            )
        )
        observed_count = len(documents)
        if observed_count >= expected_count:
            return observed_count
        time.sleep(1.0)
    return observed_count


scenario_ctx = {}

if memory_store is None:
    print("Skipped: memory_store is not initialized.")
else:
    scenario_tag = f"memory-lifecycle-{uuid.uuid4().hex[:8]}"
    now = utc_now()
    seed_specs = [
        {
            "name": "record_recall_id",
            "role": "TARGET: exact hotel-near-station and luggage condition",
            "memory": "For Osaka trips, prioritize hotels near the station that can store luggage before check-in.",
            "importance": 0.95,
            "days_old": 7,
            "access_count": 0,
        },
        {
            "name": "record_hotel_budget_id",
            "role": "SIMILAR: hotel budget condition",
            "memory": "For Osaka trips, use 20,000 JPY per night as the hotel budget guideline.",
            "importance": 0.65,
            "days_old": 4,
            "access_count": 0,
        },
        {
            "name": "record_hotel_view_id",
            "role": "SIMILAR: hotel view condition",
            "memory": "For Osaka trips, include high-rise hotels with night views as candidates.",
            "importance": 0.55,
            "days_old": 3,
            "access_count": 0,
        },
        {
            "name": "record_station_locker_id",
            "role": "SIMILAR: station/luggage but not a hotel condition",
            "memory": "For Osaka trips, also check coin lockers near the station for luggage storage.",
            "importance": 0.50,
            "days_old": 2,
            "access_count": 0,
        },
        {
            "name": "record_recent_id",
            "role": "RECENT: latest event check",
            "memory": "Check Osaka event information on the official site shortly before the trip.",
            "importance": 0.45,
            "days_old": 1,
            "access_count": 0,
        },
        {
            "name": "record_important_policy_id",
            "role": "IMPORTANT_TARGET: same text, old but important and frequently accessed",
            "memory": "The hotel policy for Osaka trips is that a takoyaki shop should be within walking distance.",
            "importance": 1.0,
            "days_old": 75,
            "access_count": 2,
        },
        {
            "name": "record_recent_note_id",
            "role": "RECENT_TARGET: same text, new but low-importance temporary note",
            "memory": "The hotel policy for Osaka trips is that a takoyaki shop should be within walking distance.",
            "importance": 0.05,
            "days_old": 0,
            "access_count": 2,
        },
        {
            "name": "record_food_id",
            "role": "DISTRACTOR: dining condition",
            "memory": "For Osaka trips, prioritize restaurants with vegetarian options.",
            "importance": 0.70,
            "days_old": 5,
            "access_count": 0,
        },
        {
            "name": "record_seat_id",
            "role": "DISTRACTOR: transit seat condition",
            "memory": "For Shinkansen rides to Osaka, prioritize aisle seats.",
            "importance": 0.60,
            "days_old": 6,
            "access_count": 0,
        },
        {
            "name": "record_transport_id",
            "role": "DISTRACTOR: local transportation condition",
            "memory": "For local transportation in Osaka, prioritize the subway and use taxis only on rainy days.",
            "importance": 0.50,
            "days_old": 8,
            "access_count": 0,
        },
        {
            "name": "record_forgetting_id",
            "role": "OLD: stale event candidate",
            "memory": "For Osaka trips, also check event candidates found six months ago just in case.",
            "importance": 0.20,
            "days_old": 140,
            "access_count": 0,
        },
        {
            "name": "record_old_hotel_id",
            "role": "OLD: stale hotel candidate",
            "memory": "For Osaka trips, include the suburban hotel used long ago as a candidate.",
            "importance": 0.25,
            "days_old": 180,
            "access_count": 0,
        },
    ]

    seed_records: dict[str, MemoryRecord] = {}
    for spec in seed_specs:
        record = MemoryRecord(
            id=str(uuid.uuid4()),
            user_id=DEMO_USER_ID,
            agent_id=DEMO_AGENT_ID,
            memory=spec["memory"],
            summary=spec["memory"],
            category=scenario_tag,
            importance=float(spec["importance"]),
            confidence=0.90,
            created_at=now - timedelta(days=int(spec["days_old"])),
            updated_at=now - timedelta(days=int(spec["days_old"])),
            access_count=int(spec["access_count"]),
            metadata={"scenario_role": spec["role"]},
        )
        record.retention_score = memory_policy.retention_score(record, now=now)
        record.status = memory_policy.status_for(record, now=now)
        seed_records[spec["name"]] = record
        memory_store.upsert(record, embedding=memory_intelligence.embed(record.memory))

    observed_count = wait_for_scenario_documents(memory_store, category=scenario_tag, expected_count=len(seed_records))
    scenario_ctx = {
        "scenario_tag": scenario_tag,
        **{name: record.id for name, record in seed_records.items()},
    }

    print("Created scenario documents.")
    print(json.dumps(
        {
            "scenario_tag": scenario_ctx["scenario_tag"],
            "expected_count": len(seed_records),
            "search_visible_count": observed_count,
            "target_memory": seed_records["record_recall_id"].memory,
            "important_target": seed_records["record_important_policy_id"].memory,
            "recent_target": seed_records["record_recent_note_id"].memory,
            "record_ids": {name: record.id for name, record in seed_records.items()},
        },
        ensure_ascii=False,
        indent=2,
    ))

### 9-1. Verify Recall Reinforcement in Scores

This cell searches for the concrete query `Osaka trip hotel near the station luggage before check-in` using `SearchMode.HYBRID` and `RankingProfile.IMPORTANT`. The scenario data includes similar memories, so the `role` field shows which result is the true target and which results are only similar.

Checkpoints:

- `RankingProfile.IMPORTANT` uses `memory-priority` internally
- The target memory is explicitly recalled and touched
- `touched_expected_target` becomes `true`
- After Azure AI Search reflects the update, only the touched memory changes `access_count`, `retention_score`, and `last_accessed_at`

In [ ]:
if "scenario_ctx" not in globals() or not scenario_ctx:
    print("Skipped: run the scenario initialization cell first.")
else:
    query = "Osaka trip hotel near the station luggage before check-in"
    query_vector = memory_intelligence.embed(query)
    priority_options = MemorySearchOptions(
        mode=SearchMode.HYBRID,
        ranking_profile=RankingProfile.IMPORTANT,
        top=5,
        category=scenario_ctx["scenario_tag"],
        include_archived=True,
    )
    expected_id = scenario_ctx["record_recall_id"]
    expected_before_touch = memory_store.get(expected_id)
    previous_access_count = expected_before_touch.access_count if expected_before_touch else -1

    results_before_touch = memory_store.search(
        query=query,
        query_vector=query_vector,
        user_id=DEMO_USER_ID,
        agent_id=DEMO_AGENT_ID,
        options=priority_options,
    )
    before_summary = summarize_scored_results(results_before_touch)

    recalled_results = [result for result in results_before_touch if result.record.id == expected_id]
    if not recalled_results and expected_before_touch:
        recalled_results = [MemorySearchResult(record=expected_before_touch)]

    memory_store.touch_results(recalled_results, memory_policy)

    expected_after_touch = None
    deadline = time.monotonic() + 5.0
    while time.monotonic() < deadline:
        candidate = memory_store.get(expected_id)
        if candidate and candidate.access_count > previous_access_count:
            expected_after_touch = candidate
            break
        time.sleep(0.5)
    if expected_after_touch is None:
        expected_after_touch = memory_store.get(expected_id)

    touched_ids = [result.record.id for result in recalled_results]
    touched_expected_target = expected_id in touched_ids
    results_after_touch = memory_store.search(
        query=query,
        query_vector=query_vector,
        user_id=DEMO_USER_ID,
        agent_id=DEMO_AGENT_ID,
        options=priority_options,
    )

    print("[1] Recall reinforcement with important ranking")
    print(json.dumps(
        {
            "mode": str(priority_options.mode),
            "ranking_profile": str(priority_options.ranking_profile),
            "query": query,
            "expected_top_role": "TARGET: exact hotel-near-station and luggage condition",
            "expected_touched_id": expected_id,
            "touched_ids": touched_ids,
            "touched_expected_target": touched_expected_target,
            "expected_before_touch": summarize_scored_results([MemorySearchResult(record=expected_before_touch)]) if expected_before_touch else [],
            "expected_after_touch": summarize_scored_results([MemorySearchResult(record=expected_after_touch)]) if expected_after_touch else [],
            "before": before_summary,
            "after": summarize_scored_results(results_after_touch),
        },
        ensure_ascii=False,
        indent=2,
    ))

### 9-2. Compare IMPORTANT and RECENT Ranking

This cell combines the same `SearchMode.HYBRID` retrieval mode with `RankingProfile.IMPORTANT` and `RankingProfile.RECENT`. The two target memories use the same search terms so ranking differences are easier to attribute to `importance`, `access_count`, and `updated_at`.

- `IMPORTANT_TARGET`: same search terms, old but important and frequently accessed
- `RECENT_TARGET`: same search terms, new but low importance

Checkpoints:

- `RankingProfile.IMPORTANT` tends to rank `IMPORTANT_TARGET` higher
- `RankingProfile.RECENT` tends to rank `RECENT_TARGET` higher
- The same retrieval mode can express different ranking intent through the ranking profile

In [ ]:
if "scenario_ctx" not in globals() or not scenario_ctx:
    print("Skipped: run the scenario initialization cell first.")
else:
    query = "Osaka trip planning policy"
    query_vector = memory_intelligence.embed(query)
    important_options = MemorySearchOptions(
        mode=SearchMode.HYBRID,
        ranking_profile=RankingProfile.IMPORTANT,
        top=10,
        category=scenario_ctx["scenario_tag"],
        include_archived=True,
    )
    recency_options = MemorySearchOptions(
        mode=SearchMode.HYBRID,
        ranking_profile=RankingProfile.RECENT,
        top=10,
        category=scenario_ctx["scenario_tag"],
        include_archived=True,
    )
    important_results = memory_store.search(
        query=query,
        query_vector=query_vector,
        user_id=DEMO_USER_ID,
        agent_id=DEMO_AGENT_ID,
        options=important_options,
    )
    recency_results = memory_store.search(
        query=query,
        query_vector=query_vector,
        user_id=DEMO_USER_ID,
        agent_id=DEMO_AGENT_ID,
        options=recency_options,
    )

    print("[2] Compare IMPORTANT and RECENT ranking profiles")
    print(json.dumps(
        {
            "query": query,
            "important_expected_role": "IMPORTANT_TARGET: same search terms, old but important and frequently recalled policy",
            "recent_expected_role": "RECENT_TARGET: same search terms, new but low-importance temporary candidate",
            "important_expected_id": scenario_ctx["record_important_policy_id"],
            "recent_expected_id": scenario_ctx["record_recent_note_id"],
            "mode": str(important_options.mode),
            "important_ranking_profile": str(important_options.ranking_profile),
            "important_results": summarize_scored_results(important_results),
            "recency_ranking_profile": str(recency_options.ranking_profile),
            "recency_results": summarize_scored_results(recency_results),
        },
        ensure_ascii=False,
        indent=2,
    ))

### 9-3. Verify Explicit Deletion in Search Results

This cell marks the target memory as `deleted`, then searches with `SearchMode.HYBRID` and `RankingProfile.IMPORTANT`.

Checkpoints:

- The target memory's `status` becomes `deleted`
- Even high-importance memories are filtered out when they are deleted

In [ ]:
if "scenario_ctx" not in globals() or not scenario_ctx:
    print("Skipped: run the scenario initialization cell first.")
else:
    memory_store.mark_status(scenario_ctx["record_recall_id"], MemoryStatus.DELETED)
    deleted_after = memory_store.get(scenario_ctx["record_recall_id"])
    visible_after_delete = memory_store.search(
        query="Osaka trip hotel priorities",
        query_vector=None,
        user_id=DEMO_USER_ID,
        agent_id=DEMO_AGENT_ID,
        options=MemorySearchOptions(
            mode=SearchMode.KEYWORD,
            ranking_profile=RankingProfile.IMPORTANT,
            top=10,
            category=scenario_ctx["scenario_tag"],
            include_archived=True,
        ),
    )

    print("[3] Explicit deletion and important ranking")
    print(json.dumps(
        {
            "deleted": {
                "id": scenario_ctx["record_recall_id"],
                "status": str(deleted_after.status) if deleted_after else None,
            },
            "mode": str(SearchMode.KEYWORD),
            "ranking_profile": str(RankingProfile.IMPORTANT),
            "visible_after_delete": summarize_scored_results(visible_after_delete),
        },
        ensure_ascii=False,
        indent=2,
    ))

### 9-4. Compare Forgetting Dry-Run and Answer-Time Search

This cell uses `apply_forgetting(..., dry_run=True)` to preview forgetting candidates without writing updates.

Forgetting candidate inspection has a different purpose than answer-time ranking. Forgetting wants weak memories, so it uses `retention_score asc`. Answer-time search uses `RankingProfile.IMPORTANT` or `RankingProfile.RECENT` to move useful memories upward.

Checkpoints:

- How scenario memories are evaluated as forgetting candidates
- Forgetting candidates differ from top results under `RankingProfile.IMPORTANT`

In [ ]:
if "scenario_ctx" not in globals() or not scenario_ctx:
    print("Skipped: run the scenario initialization cell first.")
else:
    forgetting_preview = memory_store.apply_forgetting(memory_policy, dry_run=True, limit=100)
    scenario_ids = {
        scenario_ctx["record_recall_id"],
        scenario_ctx["record_forgetting_id"],
        scenario_ctx["record_recent_id"],
    }
    scenario_preview = [item for item in forgetting_preview if item.get("id") in scenario_ids]
    important_results = memory_store.search(
        query="Osaka trip priorities",
        query_vector=None,
        user_id=DEMO_USER_ID,
        agent_id=DEMO_AGENT_ID,
        options=MemorySearchOptions(
            mode=SearchMode.KEYWORD,
            ranking_profile=RankingProfile.IMPORTANT,
            top=10,
            category=scenario_ctx["scenario_tag"],
            include_archived=True,
        ),
    )

    print("[4] Forgetting dry-run and important ranking")
    print(json.dumps(
        {
            "forgetting_preview": scenario_preview,
            "mode": str(SearchMode.KEYWORD),
            "ranking_profile": str(RankingProfile.IMPORTANT),
            "important_results": summarize_scored_results(important_results),
        },
        ensure_ascii=False,
        indent=2,
    ))

### 9-5. Check Search Visibility

This cell compares default search with a search that sets `include_archived=True`. The goal is to see how memory status affects visibility.

By default, `MemorySearchOptions` only searches `active` memories. `include_archived=True` adds `archived` memories back into the candidate set. `deleted` memories are intentionally excluded from both default and audit searches.

Checkpoints:

- `deleted` memories do not appear in search results
- `archived` memories can be inspected when explicitly included
- Agent-facing searches and operational audit searches use different `MemorySearchOptions`

In [ ]:
if "scenario_ctx" not in globals() or not scenario_ctx:
    print("Skipped: run the scenario initialization cell first.")
else:
    default_results = memory_store.search(
        query="Osaka trip",
        query_vector=memory_intelligence.embed("Osaka trip"),
        user_id=DEMO_USER_ID,
        agent_id=DEMO_AGENT_ID,
        options=MemorySearchOptions(
            mode=SearchMode.HYBRID,
            top=10,
            category=scenario_ctx["scenario_tag"],
        ),
    )
    include_archived_results = memory_store.search(
        query="Osaka trip",
        query_vector=memory_intelligence.embed("Osaka trip"),
        user_id=DEMO_USER_ID,
        agent_id=DEMO_AGENT_ID,
        options=MemorySearchOptions(
            mode=SearchMode.HYBRID,
            top=10,
            category=scenario_ctx["scenario_tag"],
            include_archived=True,
        ),
    )

    print("[5] Search visibility and hybrid search")
    print(json.dumps(
        {
            "mode": str(SearchMode.HYBRID),
            "default_results": summarize_scored_results(default_results),
            "include_archived_results": summarize_scored_results(include_archived_results),
            "deleted_id": scenario_ctx["record_recall_id"],
            "decayed_id": scenario_ctx["record_forgetting_id"],
            "recent_id": scenario_ctx["record_recent_id"],
        },
        ensure_ascii=False,
        indent=2,
    ))

## 10. Agent Framework Implementation Example

In a real app, pass `custom_memory_provider` to an agent's `context_providers`. `before_run()` searches Azure AI Search for relevant memories and injects them into context. `after_run()` extracts memory candidates from the conversation and handles add, update, and delete operations.

This cell displays the result of `agent.run()` and then searches memory again on the same theme. The `tracer.start_as_current_span(...)` blocks wrap the agent run and memory checks, so Jaeger or another OTLP destination can show the memory flow.

In [ ]:
import os
from agent_framework.openai import OpenAIChatCompletionClient

mcp_server_uri = os.getenv("MCP_SERVER_URI")
azure_openai_key = os.getenv("AZURE_OPENAI_API_KEY")
azure_openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_deployment = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")
api_version = os.getenv("AZURE_OPENAI_API_VERSION")

# === Authentication mode ===
# True: API key auth, False: DefaultAzureCredential auth
USE_KEY_AUTH = os.getenv("USE_KEY_AUTH", "False").lower() in ("true", "1", "t")

if USE_KEY_AUTH:
    auth_kwargs = dict(api_key=azure_openai_key)
    print("Auth mode: API key")
else:
    from azure.identity.aio import DefaultAzureCredential
    auth_kwargs = dict(credential=DefaultAzureCredential())
    # The framework may automatically read AZURE_OPENAI_API_KEY from the environment
    # and prefer it over credential auth, so remove it explicitly
    os.environ.pop("AZURE_OPENAI_API_KEY", None)
    print("Auth mode: DefaultAzureCredential")

project_endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model_deployment = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

# NOTE: The agent is used with `async with`, so create a fresh client for each run.
def create_azure_openai_chat_client():
    return OpenAIChatCompletionClient(
        **auth_kwargs,
        azure_endpoint=azure_openai_endpoint,
        api_version=api_version,
        model=azure_deployment
    )

print(f"MCP Server URI: {mcp_server_uri}")
print(f"Foundry Project Endpoint: {project_endpoint}")
print(f"Foundry Model Deployment: {model_deployment}")
print(f"Azure OpenAI Endpoint: {azure_openai_endpoint}, Deployment: {azure_deployment}")

In [ ]:
print("Agent Framework wiring example")

from agent_framework import Agent


def render_message_text(message: Any) -> str:
    text = getattr(message, "text", None)
    if text:
        return str(text)
    contents = getattr(message, "contents", None)
    if contents:
        return "\n".join(str(item) for item in contents)
    content = getattr(message, "content", None)
    if content:
        return str(content)
    return str(message)


def render_agent_result(result: Any) -> str:
    text = getattr(result, "text", None)
    if text:
        return str(text)
    messages = getattr(result, "messages", None)
    if messages:
        return "\n".join(render_message_text(message) for message in messages)
    value = getattr(result, "value", None)
    if value is not None and value is not result:
        return render_agent_result(value)
    content = getattr(result, "content", None)
    if content:
        return str(content)
    return str(result)


def summarize_agent_memory_results(results: list[MemorySearchResult]) -> list[dict[str, Any]]:
    return [
        {
            "rank": index,
            "score": result.score,
            "reranker_score": result.reranker_score,
            "memory": result.record.memory,
            "category": result.record.category,
            "importance": result.record.importance,
            "retention_score": result.record.retention_score,
            "access_count": result.record.access_count,
            "status": str(result.record.status),
            "updated_at": dt_to_json(result.record.updated_at),
        }
        for index, result in enumerate(results, start=1)
    ]


if custom_memory_provider is None:
    print("Skipped: custom_memory_provider is not initialized.")
elif not azure_openai_endpoint or not azure_deployment or not api_version:
    print("Skipped: Azure OpenAI endpoint, deployment, or api_version is missing.")
else:
    user_query = "Plan my Osaka trip"
    agent = Agent(
        client=create_azure_openai_chat_client(),
        name="CustomMemoryTravelAgent",
        instructions="Use relevant memories only when they help answer the user.",
        context_providers=[custom_memory_provider],
    )

    agent_session = agent.create_session()
    with tracer.start_as_current_span("custom_memory_agent_run_plan") as span:
        span.set_attribute("agent.name", "CustomMemoryTravelAgent")
        span.set_attribute("user.query", user_query)
        span.set_attribute("memory.search.mode", str(custom_memory_provider.search_options.mode))
        span.set_attribute("memory.search.top", custom_memory_provider.search_options.top)
        agent_result = await agent.run(user_query, session=agent_session)

    print("\n[1] Agent response")
    print(render_agent_result(agent_result))


In [ ]:
if custom_memory_provider is None:
    print("Skipped: custom_memory_provider is not initialized.")
elif not azure_openai_endpoint or not azure_deployment or not api_version:
    print("Skipped: Azure OpenAI endpoint, deployment, or api_version is missing.")
else:
    user_query = "What should I prioritize for my Osaka trip?"
    agent = Agent(
        client=create_azure_openai_chat_client(),
        name="CustomMemoryTravelAgent",
        instructions="Use relevant memories only when they help answer the user.",
        context_providers=[custom_memory_provider],
    )

    agent_session = agent.create_session()
    with tracer.start_as_current_span("custom_memory_agent_run_priority_question") as span:
        span.set_attribute("agent.name", "CustomMemoryTravelAgent")
        span.set_attribute("user.query", user_query)
        span.set_attribute("memory.search.mode", str(custom_memory_provider.search_options.mode))
        span.set_attribute("memory.search.top", custom_memory_provider.search_options.top)
        agent_result = await agent.run(user_query, session=agent_session)

    print("\n[1] Agent response")
    print(render_agent_result(agent_result))


## 11. Summary

Custom memory means you own several responsibilities. However, ranking that belongs in the search engine should be modeled in Azure AI Search instead of being bolted on in Python.

| Responsibility | Class in this notebook | What to inspect |
|---|---|---|
| Extract memory candidates from conversation | `MemoryExtractor` | Which utterances should become memories |
| Consolidate with existing memories | `MemoryConsolidator` | How add, update, delete, and no-op decisions are separated |
| Generate embeddings | `AzureOpenAIMemoryIntelligence.embed()` | How fallback behaves when Azure OpenAI is unavailable |
| Search, save, filter, and manage status | `AzureAISearchMemoryStore` | How Azure AI Search fields become search conditions |
| Adjust ranking | Azure AI Search scoring profiles | How `RankingProfile.IMPORTANT` and `RankingProfile.RECENT` select ranking rules |
| Specify search policy | `MemorySearchOptions` | Which memories the agent sees, and which retrieval/ranking policy is used |
| Inject and save around agent execution | `AzureAISearchMemoryContextProvider` | Recall before the run and memory update after the run |
| Recall, forgetting, and old memories | `HumanMemoryPolicy` | Strengthen recalled memories and weaken old ones |

The strength of this design is that the Azure AI Search schema is designed directly for memory. By promoting dates, importance, status, access count, and expiration to fields, filters, sorting, scoring profiles, and semantic ranking become natural parts of the memory layer.

The first production knobs to tune are `mode`, `ranking_profile`, `top`, `min_retention_score`, and `include_archived`. Reduce `top` or raise retention filters when irrelevant memories appear. Use `RankingProfile.IMPORTANT` when durable preferences should dominate, and `RankingProfile.RECENT` when recent updates should dominate.

## Reference
### 9-0. When Are the Four Score Fields Updated?

`importance`, `retention_score`, `access_count`, and `updated_at` all affect retrieval and memory lifetime, but they serve different purposes. In particular, `updated_at` means when the memory itself was added, updated, or deleted. Recall time is stored separately in `last_accessed_at`.

| Field | Updated when | Logic | Main use |
|---|---|---|---|
| `importance` | When extracting a memory candidate or merging into an existing memory | LLM extraction returns an `importance` value normalized by `coerce_unit_score()`. Rule fallback assigns `0.65` to durable-looking preferences or constraints. `UPDATE` keeps `max(existing.importance, candidate.importance)`. | Feeds `retention_score` and `RankingProfile.IMPORTANT` |
| `retention_score` | `ADD`, `UPDATE`, recall through `touch_results()`, and forgetting checks | `HumanMemoryPolicy.retention_score()` returns `0.0` when expired. Otherwise it clamps `decay * importance_anchor + access_rehearsal` to 0.0-1.0. | Filters weak memories and contributes to `RankingProfile.IMPORTANT` |
| `access_count` | When a memory is actually used by pre-run retrieval | `before_run()` searches memories and passes only retrieved memories to `store.touch_results()`, where `policy.touch()` increments access count. Direct debug calls to `search_memories()` do not increment it. | Helps frequently recalled memories survive and rank higher with `RankingProfile.IMPORTANT` |
| `updated_at` | When a memory is added, merged, or marked deleted | `ADD` sets `created_at` and `updated_at` to now. `UPDATE` refreshes `updated_at`. `DELETE` updates status and timestamp. Recall updates `last_accessed_at`, not `updated_at`. | Freshness signal for `RankingProfile.RECENT` |

At runtime, `before_run()` searches relevant memories and updates only those memories that were actually used. `after_run()` extracts candidates from the latest conversation and decides `ADD` / `UPDATE` / `DELETE` / `NONE`. Added and updated memories receive final `importance`, `retention_score`, and `updated_at` values.

In Azure AI Search, `RankingProfile.IMPORTANT` boosts `importance`, `retention_score`, and `access_count`. `RankingProfile.RECENT` emphasizes `updated_at`. Python updates lifecycle fields, while Azure AI Search applies the final ranking logic.

#### About the `importance` Score

Azure AI Search does not decide `importance`; it only uses the stored value for ranking. The value is produced by `MemoryExtractor`. With an LLM, the model returns importance alongside each memory candidate as JSON. Without an LLM, rule fallback assigns a fixed value to durable preferences, priorities, dislikes, or requirements. During `UPDATE`, the consolidator keeps the higher value from the existing memory and the new candidate.

#### About `updated_at` and `retention_score`

`updated_at` and `retention_score` answer different questions. `updated_at` asks, "Was this memory changed recently?" `retention_score` asks, "Is this memory still worth keeping and using?" A six-month-old preference such as "prefer vegetarian-friendly restaurants" can still be important, while a fresh note such as "tomorrow's meeting is at 10" can become weak once it expires.

Relying only on `updated_at` can overvalue temporary new notes and undervalue old durable preferences. `retention_score` combines importance, age, recall count, and expiration, giving Azure AI Search a lifecycle-aware signal beyond freshness alone.

## Human-Like Memory: HumanMemoryPolicy

As an agent's memory grows, old plans and weak preferences can reduce answer quality if they are always retrieved. `HumanMemoryPolicy` keeps recalled memories easier to find and gradually fades memories that are not used.

### Retention Score Formula (Simplified)

A richer cognitive model is possible, but this notebook uses a simple model suitable for demonstration.

The retention score represents, from 0.0 to 1.0, whether a memory is still worth showing to the agent. Low scores can be filtered out, and scores below the threshold move memories to `archived`.

The score combines three independent factors:

$$
\text{retention\_score} = \underbrace{\text{decay} \times \text{importance\_anchor}}_{\text{base value from time and importance}} + \underbrace{\text{access\_rehearsal}}_{\text{recall reinforcement}}
$$

The product on the left captures time decay and importance. The additive term captures reinforcement from actual use.

---

#### Element 1: decay

$$
\text{decay} = 0.5^{\frac{\text{age\_days}}{\text{half\_life\_days}}}
$$

`decay` measures how much freshness remains based on days since the memory was updated. The default half-life is 45 days, so the value halves after 45 days, becomes one quarter after 90 days, and falls to about 0.06 after 180 days.

By itself, this would fade durable preferences and temporary notes at the same rate. The next term adjusts for importance.

---

#### Element 2: importance_anchor

$$
\text{importance\_anchor} = 0.35 + 0.65 \times \text{importance}
$$

`importance_anchor` is multiplied by decay. High-importance memories keep a higher score for the same age.

For `importance = 1.0`, the anchor is 1.0. For `importance = 0.0`, the anchor is 0.35. For example, after 90 days, decay is 0.25. A high-importance memory remains at 0.25, while a zero-importance memory drops to 0.088.

The 0.35 floor prevents low-importance memories from disappearing immediately after creation.

---

#### Element 3: access_rehearsal

$$
\text{access\_rehearsal} = \min(\text{access\_count} \times 0.04,\ 0.35)
$$

`access_rehearsal` adds reinforcement for memories that are actually retrieved before agent execution. Each use adds 0.04 up to a maximum of 0.35.

Because this term is additive, an old memory with near-zero decay can still remain visible if it has been recalled often. This mirrors the intuition that information used repeatedly is easier to retain.

---

#### How the Three Elements Combine

1. Compute freshness from elapsed days (`decay`).
2. Weight that freshness by importance (`decay × anchor`).
3. Add reinforcement from actual use (`+ rehearsal`).

This creates three useful behaviors:

**New and important memories** score high even before they are recalled.

**Old but frequently used memories** stay visible because recall reinforcement offsets time decay.

**Old and unused memories** naturally fall below the archive threshold, especially when importance is low.

---

#### Parameters

| Parameter | Default | Meaning |
|---|---|---|
| `half_life_days` | 45.0 | Time-decay half-life. Larger values make memories last longer |
| `access_boost` | 0.04 | Reinforcement per recall, capped at 0.35 |
| `archive_threshold` | 0.22 | Memories at or below this score become `archived` and are excluded by default |

### Recall Reinforcement

Memories used by `before_run()` are updated by `touch_results()`:

```python
record.last_accessed_at = now
record.access_count += 1
record.retention_score = policy.retention_score(record)
record.status = policy.status_for(record)
```

Frequently used memories remain easier to find, while unused memories gradually fade out. This is a simplified Ebbinghaus-style forgetting curve.

### Update Timing for the Four Score Fields

| Field | Updated when | Search impact |
|---|---|---|
| `importance` | Memory extraction or consolidation into an existing memory | `RankingProfile.IMPORTANT` ranking signal |
| `retention_score` | ADD / UPDATE / recall / forgetting check | Filter signal and `RankingProfile.IMPORTANT` ranking signal |
| `access_count` | When a `before_run()` search result is passed to `touch_results()` | `RankingProfile.IMPORTANT` ranking signal |
| `updated_at` | ADD / UPDATE / DELETE, but not recall | Freshness signal for `RankingProfile.RECENT` |